# Lab 07: Neural Network (Computer Vision)

| | |
|---|---|
| **Name** | Muhammad Talha Ramzan |
| **Roll No** | 2330-0141 |
| **Course** | Computer Vision (CV) |
| **Lab** | Lab 07 - Neural Network |

---
## Lab Outcomes
- Explain the structure of a fully connected MLP (weights, biases, activations).
- Perform a numerical forward pass using ReLU, tanh and sigmoid.
- Calculate binary cross-entropy (BCE) loss.
- Derive and implement backpropagation using the chain rule.
- Update parameters with one gradient-descent step and verify gradients independently.

## 1. Network Architecture
2 inputs -> 2 hidden units (activation g) -> 1 output (sigmoid)

**Forward pass**

- z1 = W1 x + b1
- h = g(z1)
- z2 = W2 h + b2
- y = sigmoid(z2)

**Loss (BCE):** L = -[t ln(y) + (1 - t) ln(1 - y)], and for sigmoid + BCE, dL/dz2 = y - t.

## 2. Forward Propagation, Loss, Backpropagation and One Gradient-Descent Update (NumPy)

In [1]:
import numpy as np
import pandas as pd

np.set_printoptions(precision=6, suppress=True)

# Input and target
x = np.array([[1.0], [-2.0]])
t = 1.0   # assumed ground-truth label

# Parameters: 2 inputs -> 2 hidden -> 1 output
W1 = np.array([[0.5, -0.5],
               [-1.0, 2.0]])
b1 = np.array([[0.0], [1.0]])
W2 = np.array([[2.0, 1.0]])
b2 = -1.0

def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

EPS = 1e-12

# ---------- Forward propagation ----------
z1 = W1 @ x + b1
h = np.maximum(0.0, z1)            # ReLU
z2 = (W2 @ h).item() + b2
y = sigmoid(z2)

y_safe = np.clip(y, EPS, 1.0 - EPS)
loss = -(t * np.log(y_safe) + (1 - t) * np.log(1 - y_safe))

print("z1 =", z1.ravel())
print("ReLU hidden h =", h.ravel())
print("z2 =", z2)
print("prediction y =", y)
print("Predicted class (threshold 0.5) =", int(y >= 0.5))
print("BCE loss =", loss)

z1 = [ 1.5 -4. ]
ReLU hidden h = [1.5 0. ]
z2 = 2.0
prediction y = 0.8807970779778823
Predicted class (threshold 0.5) = 1
BCE loss = 0.12692801104297263


In [2]:
# ---------- Backpropagation ----------
dz2 = y - t                      # sigmoid + BCE
dW2 = dz2 * h.T
db2 = dz2
dh = W2.T * dz2
relu_mask = (z1 > 0).astype(float)
dz1 = dh * relu_mask
dW1 = dz1 @ x.T
db1 = dz1

print("dL/dz2 =", dz2)
print("dL/dh  =", dh.ravel())
print("ReLU mask =", relu_mask.ravel())
print("dW2 =\n", dW2)
print("db2 =", db2)
print("dW1 =\n", dW1)
print("db1 =\n", db1)

dL/dz2 = -0.11920292202211769
dL/dh  = [-0.238406 -0.119203]
ReLU mask = [1. 0.]
dW2 =
 [[-0.178804 -0.      ]]
db2 = -0.11920292202211769
dW1 =
 [[-0.238406  0.476812]
 [ 0.        0.      ]]
db1 =
 [[-0.238406]
 [-0.      ]]


In [3]:
# ---------- One gradient-descent update ----------
learning_rate = 0.1
W1_new = W1 - learning_rate * dW1
b1_new = b1 - learning_rate * db1
W2_new = W2 - learning_rate * dW2
b2_new = b2 - learning_rate * db2

print("Updated W1 =\n", W1_new)
print("Updated b1 =\n", b1_new)
print("Updated W2 =", W2_new)
print("Updated b2 =", b2_new)

# New forward pass with updated parameters
z1n = W1_new @ x + b1_new
hn = np.maximum(0.0, z1n)
z2n = (W2_new @ hn).item() + b2_new
yn = sigmoid(z2n)
loss_new = -(t * np.log(yn) + (1 - t) * np.log(1 - yn))
print(f"\nLoss before update = {loss:.6f}")
print(f"Loss after  update = {loss_new:.6f}  (decreased: {loss_new < loss})")

Updated W1 =
 [[ 0.523841 -0.547681]
 [-1.        2.      ]]
Updated b1 =
 [[0.023841]
 [1.      ]]
Updated W2 = [[2.01788 1.     ]]
Updated b2 = -0.9880797077977882

Loss before update = 0.126928
Loss after  update = 0.093081  (decreased: True)


## 3. Compare ReLU and tanh Hidden Activations

In [4]:
h_relu = np.maximum(0.0, z1)
h_tanh = np.tanh(z1)

print("Pre-activation z1:", z1.ravel())
print("ReLU output:", h_relu.ravel())
print("tanh output:", h_tanh.ravel())

z2_tanh = (W2 @ h_tanh).item() + b2
print("Output probability with tanh:", sigmoid(z2_tanh))

Pre-activation z1: [ 1.5 -4. ]
ReLU output: [1.5 0. ]
tanh output: [ 0.905148 -0.999329]
Output probability with tanh: 0.4528820255037602


## 4. Reusable Functions (for the experiments)
A general forward/backward implementation that supports ReLU, tanh and sigmoid hidden activations.

In [5]:
ACT = {
    "relu":    (lambda z: np.maximum(0.0, z),  lambda z: (z > 0).astype(float)),
    "tanh":    (np.tanh,                        lambda z: 1 - np.tanh(z) ** 2),
    "sigmoid": (sigmoid,                        lambda z: sigmoid(z) * (1 - sigmoid(z))),
}

def forward(x, W1, b1, W2, b2, act="relu"):
    g, _ = ACT[act]
    z1 = W1 @ x + b1
    h = g(z1)
    z2 = (W2 @ h).item() + b2
    y = sigmoid(z2)
    return z1, h, z2, y

def bce(y, t):
    y = np.clip(y, EPS, 1 - EPS)
    return -(t * np.log(y) + (1 - t) * np.log(1 - y))

def backward(x, t, z1, h, y, W2, act="relu"):
    _, dg = ACT[act]
    dz2 = y - t
    dW2 = dz2 * h.T
    db2 = dz2
    dh = W2.T * dz2
    dz1 = dh * dg(z1)
    dW1 = dz1 @ x.T
    db1 = dz1
    return dW1, db1, dW2, db2

def step(params, grads, lr):
    return tuple(p - lr * g for p, g in zip(params, grads))

# sanity check against the worked example
z1_, h_, z2_, y_ = forward(x, W1, b1, W2, b2, "relu")
print("y =", y_, "| loss =", bce(y_, 1.0))

y = 0.8807970779778823 | loss = 0.12692801104297263


## Experiment 1: Replace ReLU with tanh

In [6]:
for act in ["relu", "tanh"]:
    z1_, h_, z2_, y_ = forward(x, W1, b1, W2, b2, act)
    print(f"{act:5s} | z1 = {z1_.ravel()} | h = {h_.ravel()} | z2 = {z2_:.6f} | y = {y_:.6f} | loss(t=1) = {bce(y_, 1.0):.6f}")

relu  | z1 = [ 1.5 -4. ] | h = [1.5 0. ] | z2 = 2.000000 | y = 0.880797 | loss(t=1) = 0.126928
tanh  | z1 = [ 1.5 -4. ] | h = [ 0.905148 -0.999329] | z2 = -0.189033 | y = 0.452882 | loss(t=1) = 0.792124


**Observation:** The pre-activation z1 = [1.5, -4] is the same for both. ReLU gives h = [1.5, 0] (the negative unit is switched off), while tanh gives h = [0.905, -0.999] (squashed into (-1, 1), keeping the negative unit active with a negative value). Because h2 is negative under tanh, it *reduces* z2, so the output probability drops from 0.8808 (ReLU) to 0.4529 (tanh), the predicted class flips from 1 to 0 at the 0.5 threshold, and the loss rises from 0.1269 to 0.7921 for t = 1.

## Experiment 2: Change the target from t = 1 to t = 0

In [7]:
z1_, h_, z2_, y_ = forward(x, W1, b1, W2, b2, "relu")
for tt in [1.0, 0.0]:
    g = backward(x, tt, z1_, h_, y_, W2, "relu")
    print(f"--- t = {tt} ---")
    print("loss =", bce(y_, tt))
    print("dL/dz2 =", y_ - tt)
    print("dW1 =\n", g[0]); print("db1 =", g[1].ravel())
    print("dW2 =", g[2]); print("db2 =", g[3])

--- t = 1.0 ---
loss = 0.12692801104297263
dL/dz2 = -0.11920292202211769
dW1 =
 [[-0.238406  0.476812]
 [ 0.        0.      ]]
db1 = [-0.238406 -0.      ]
dW2 = [[-0.178804 -0.      ]]
db2 = -0.11920292202211769
--- t = 0.0 ---
loss = 2.1269280110429714
dL/dz2 = 0.8807970779778823
dW1 =
 [[ 1.761594 -3.523188]
 [ 0.        0.      ]]
db1 = [1.761594 0.      ]
dW2 = [[1.321196 0.      ]]
db2 = 0.8807970779778823


**Observation:** With t = 0 the model is confidently wrong (y = 0.88), so the loss jumps to about 2.127 and dL/dz2 = +0.8808 (versus -0.1192 for t = 1). The gradients flip sign and become much larger, so the update pushes the output *down* instead of up. The second hidden unit still gets zero gradient because of the ReLU mask.

## Experiment 3: Learning rate 0.01, 0.1 and 0.5

In [8]:
params = (W1, b1, W2, b2)
z1_, h_, z2_, y_ = forward(x, W1, b1, W2, b2, "relu")
grads = backward(x, 1.0, z1_, h_, y_, W2, "relu")
print(f"Loss before update: {bce(y_, 1.0):.6f}\n")

rows = []
for lr in [0.01, 0.1, 0.5]:
    nW1, nb1, nW2, nb2 = step(params, grads, lr)
    _, hn_, _, yn_ = forward(x, nW1, nb1, nW2, nb2, "relu")
    rows.append({"lr": lr, "W1": np.round(nW1, 4).tolist(), "b1": np.round(nb1.ravel(), 4).tolist(),
                 "W2": np.round(nW2, 4).tolist(), "b2": round(float(nb2), 4),
                 "new y": round(float(yn_), 6), "new loss": round(float(bce(yn_, 1.0)), 6)})
pd.DataFrame(rows)

Loss before update: 0.126928



,lr,W1,b1,W2,b2,new y,new loss
0,0.01,"[[0.5024, -0.5048], [-1.0, 2.0]]","[0.0024, 1.0]","[[2.0018, 1.0]]",-0.9988,0.884168,0.123108
1,0.10,"[[0.5238, -0.5477], [-1.0, 2.0]]","[0.0238, 1.0]","[[2.0179, 1.0]]",-0.9881,0.911120,0.093081
2,0.50,"[[0.6192, -0.7384], [-1.0, 2.0]]","[0.1192, 1.0]","[[2.0894, 1.0]]",-0.9404,0.975591,0.024712


**Observation:** A larger learning rate moves the parameters further and lowers the loss more in one step (0.5 reduces it the most here). A very small rate (0.01) changes the parameters only slightly. With a bigger or less well-behaved problem, a rate that is too large can overshoot and make the loss rise.

## Experiment 4: One positive and one negative pre-activation (ReLU mask)

In [9]:
# Flip which unit is active: x = [-1, 2] gives z1 = [-1.5, 6]
x4 = np.array([[-1.0], [2.0]])
z1_, h_, z2_, y_ = forward(x4, W1, b1, W2, b2, "relu")
dW1_4, db1_4, dW2_4, db2_4 = backward(x4, 1.0, z1_, h_, y_, W2, "relu")
print("z1 =", z1_.ravel())
print("ReLU mask =", (z1_ > 0).astype(float).ravel())
print("h =", h_.ravel(), "| y =", y_, "| loss =", bce(y_, 1.0))
print("dW1 =\n", dW1_4)
print("db1 =", db1_4.ravel())
print("dW2 =", dW2_4)

z1 = [-1.5  6. ]
ReLU mask = [0. 1.]
h = [0. 6.] | y = 0.9933071490757153 | loss = 0.006715348489117944
dW1 =
 [[ 0.        0.      ]
 [ 0.006693 -0.013386]]
db1 = [-0.       -0.006693]
dW2 = [[-0.       -0.040157]]


**Observation:** The unit with a positive z receives gradient; the unit with a negative z is masked to zero. For x = [-1, 2], unit 1 is inactive, so row 1 of dW1 and db1[0] are zero and only unit 2 learns. In the original example it was the opposite (unit 2 inactive). A permanently negative unit stops learning, which is the "dying ReLU" problem.

## Experiment 5: Verify gradients (independent implementation)

### 5a. Finite-difference check in NumPy (runs anywhere)

In [10]:
def loss_fn(W1_, b1_, W2_, b2_):
    return bce(forward(x, W1_, b1_, W2_, b2_, "relu")[3], 1.0)

def num_grad(f, P, eps=1e-6):
    P = np.array(P, dtype=float); g = np.zeros_like(P)
    it = np.nditer(P, flags=["multi_index"])
    for _ in it:
        i = it.multi_index; old = P[i]
        P[i] = old + eps; fp = f(P)
        P[i] = old - eps; fm = f(P)
        P[i] = old; g[i] = (fp - fm) / (2 * eps)
    return g

nW1 = num_grad(lambda P: loss_fn(P, b1, W2, b2), W1)
nb1 = num_grad(lambda P: loss_fn(W1, P, W2, b2), b1)
nW2 = num_grad(lambda P: loss_fn(W1, b1, P, b2), W2)
nb2 = num_grad(lambda P: loss_fn(W1, b1, W2, float(P)), np.array(b2)).item()

diffs = [np.abs(nW1 - dW1).max(), np.abs(nb1 - db1).max(),
         np.abs(nW2 - dW2).max(), abs(nb2 - db2)]
print("Max abs diff  W1, b1, W2, b2:", diffs)
print("Overall max absolute difference:", max(diffs))

Max abs diff  W1, b1, W2, b2: [np.float64(3.9109604443865464e-11), np.float64(8.200773393696181e-12), np.float64(8.594785994020526e-11), np.float64(3.059408282268805e-11)]
Overall max absolute difference: 8.594785994020526e-11


### 5b. PyTorch autograd check (as in the lab manual)
Run this cell in your own Jupyter / Colab environment where PyTorch is installed.

In [11]:
try:
    import torch

    xt = torch.tensor([[1.0], [-2.0]], dtype=torch.float64)
    tt = torch.tensor(1.0, dtype=torch.float64)
    W1t = torch.tensor([[0.5, -0.5], [-1.0, 2.0]], dtype=torch.float64, requires_grad=True)
    b1t = torch.tensor([[0.0], [1.0]], dtype=torch.float64, requires_grad=True)
    W2t = torch.tensor([[2.0, 1.0]], dtype=torch.float64, requires_grad=True)
    b2t = torch.tensor(-1.0, dtype=torch.float64, requires_grad=True)

    z1t = W1t @ xt + b1t
    ht = torch.relu(z1t)
    z2t = (W2t @ ht).squeeze() + b2t
    loss_t = torch.nn.functional.binary_cross_entropy_with_logits(z2t, tt)
    loss_t.backward()

    print("Loss:", loss_t.item())
    print("dW1:\n", W1t.grad)
    print("db1:\n", b1t.grad)
    print("dW2:\n", W2t.grad)
    print("db2:", b2t.grad)

    d = max((W1t.grad.numpy() - dW1).__abs__().max(), (b1t.grad.numpy() - db1).__abs__().max(),
            (W2t.grad.numpy() - dW2).__abs__().max(), abs(b2t.grad.item() - db2))
    print("Max absolute difference vs NumPy:", d)
except Exception as e:
    print(f"PyTorch is not available in this environment ({type(e).__name__}). Install with `pip install torch` and re-run this cell.")

PyTorch is not available in this environment (OSError). Install with `pip install torch` and re-run this cell.


## Observation Table

In [12]:
def run(act, tt, lr=0.1, x_=x):
    z1_, h_, z2_, y_ = forward(x_, W1, b1, W2, b2, act)
    L = bce(y_, tt)
    g = backward(x_, tt, z1_, h_, y_, W2, act)
    nW1, nb1_, nW2, nb2_ = step((W1, b1, W2, b2), g, lr)
    yn_ = forward(x_, nW1, nb1_, nW2, nb2_, act)[3]
    return h_.ravel(), y_, L, bce(yn_, tt)

r1 = run("relu", 1.0)
r2 = run("tanh", 1.0)
r3 = run("relu", 0.0)
r4 = run("relu", 1.0, lr=0.5)

table = pd.DataFrame([
    ["ReLU; t = 1",             np.round(r1[0], 5), round(r1[1], 6), round(r1[2], 6),
     f"Negative unit is zeroed; correct and confident, low loss (after update: {r1[3]:.4f})"],
    ["tanh; t = 1",             np.round(r2[0], 5), round(r2[1], 6), round(r2[2], 6),
     f"Both units active; h2 negative lowers y, loss higher than ReLU (after update: {r2[3]:.4f})"],
    ["ReLU; t = 0",             np.round(r3[0], 5), round(r3[1], 6), round(r3[2], 6),
     f"Confidently wrong prediction gives large loss and large gradients (after update: {r3[3]:.4f})"],
    ["ReLU; alternate lr = 0.5", np.round(r4[0], 5), round(r4[1], 6), round(r4[2], 6),
     f"Bigger step cuts loss more in one update: {r4[2]:.4f} -> {r4[3]:.4f}"],
], columns=["Setting", "Hidden h", "Output probability y", "Loss", "Key observation"])
table

,Setting,Hidden h,Output probability y,Loss,Key observation
0,ReLU; t = 1,"[1.5, 0.0]",0.880797,0.126928,Negative unit is zeroed; correct and confident...
1,tanh; t = 1,"[0.90515, -0.99933]",0.452882,0.792124,"Both units active; h2 negative lowers y, loss ..."
2,ReLU; t = 0,"[1.5, 0.0]",0.880797,2.126928,Confidently wrong prediction gives large loss ...
3,ReLU; alternate lr = 0.5,"[1.5, 0.0]",0.880797,0.126928,Bigger step cuts loss more in one update: 0.12...


## Conclusion
- Implemented a 2-2-1 MLP forward pass with ReLU, tanh and sigmoid.
- Computed BCE loss and derived gradients by hand using the chain rule (sigmoid + BCE gives dL/dz2 = y - t).
- Performed a gradient-descent update and confirmed the loss decreased.
- Verified the analytic gradients against an independent numerical (and PyTorch) implementation.
- Saw how the ReLU derivative mask blocks gradients for inactive units.

**Muhammad Talha Ramzan | 2330-0141 | Lab 07 - CV**